# LinQ TPU: детекции YOLOv8s по видео → JSONL (schema v2) для backend

Тонкая обёртка над `run_linq_detections.py`: проверка окружения → запуск → прогресс → сводка →
ссылки на скачивание. Вся логика (letterbox, декодирование выхода, формат записи, протокол LinQ) —
в общих файлах проекта, здесь она не копируется.

**Загрузите в эту папку Jupyter (кнопка Upload):**
- `linq/run_linq_detections.py`
- `backend/app/pipeline/detectors/yolo_core.py`
- `backend/app/pipeline/detectors/replay_format.py`
- `backend/app/pipeline/detectors/linq_transport.py`
- видео (имена — как в ячейке параметров).

Программа TPU (`*.tpu`) должна лежать в `/auto/cache/<логин>/` — как в `examples/example.ipynb`.

Учётные данные сервиса LinQ вводятся в ячейке 2 и живут только в памяти ядра: **не сохраняйте их в
notebook и не коммитьте**. Перед отправкой notebook в репозиторий — «Clear All Outputs».

## 1. Проверка окружения

In [ ]:
import importlib
import os
from pathlib import Path

need = ["run_linq_detections.py", "yolo_core.py", "replay_format.py", "linq_transport.py"]
missing = [n for n in need if not Path(n).is_file()]
print("Файлы:", "все на месте" if not missing else f"НЕ ХВАТАЕТ {missing}")
for module in ("numpy", "cv2", "pika", "tpu_pycloud_format"):
    try:
        importlib.import_module(module)
        print(f"  {module}: ok")
    except ImportError as exc:
        print(f"  {module}: НЕТ ({exc}) — без него запуск на LinQ невозможен")

## 2. Параметры и подключение (значения не сохраняются в файл)

In [ ]:
import getpass

# Программа TPU, скомпилированная из YOLOv8s (полный путь в /auto/cache/<логин>/…)
PROGRAM = "/auto/cache/<логин>/<программа>.tpu"  # ЗАМЕНИТЬ
# Видео, загруженные в эту папку
VIDEOS = ["0_te1.mp4"]  # ЗАМЕНИТЬ на свои
STEP = 1  # каждый кадр (backend сам выберет нужные по analysis_fps)
WARMUP = 5  # прогревочные вызовы, в статистику не входят

if not os.environ.get("LINQ_API_URL"):
    os.environ["LINQ_API_URL"] = input("Адрес сервиса LinQ (amqp://<адрес из инструкции>:5672/): ")
if not os.environ.get("LINQ_LOGIN"):
    os.environ["LINQ_LOGIN"] = input("Логин сервиса LinQ (RabbitMQ, не Jupyter): ")
if not os.environ.get("LINQ_PASSWORD"):
    os.environ["LINQ_PASSWORD"] = getpass.getpass("Пароль сервиса LinQ: ")
print("Программа есть:", Path(PROGRAM).is_file())
print("Видео:", {v: Path(v).is_file() for v in VIDEOS})

## 3. Запуск (можно прервать и запустить снова — продолжит с последнего кадра)

In [ ]:
import run_linq_detections as gen

runtime = gen.LinqRuntime(PROGRAM, float(os.environ.get("LINQ_TIMEOUT_S", "60")))
results = {}
try:
    for video in VIDEOS:
        out = Path(f"dets_linq_{Path(video).stem}.jsonl")
        print(f"=== {video} → {out}")
        results[video] = (out, gen.run(Path(video), out, runtime, model=gen.model_meta(PROGRAM),
                                       step=STEP, warmup=WARMUP, log=print))
finally:
    runtime.close()

## 4. Сводка замеров (warm-up исключён; inference_ms — вызов runtime с ожиданием результата)

In [ ]:
for video, (_out, s) in results.items():
    inf, frame = s["inference_ms"], s["frame_ms"]
    print(f"{video}: кадров {s['frames_written']} из {s['frame_count_read']} прочитанных; "
          f"вызов runtime avg {inf['avg']} / p50 {inf['p50']} / p95 {inf['p95']} мс; "
          f"полное время кадра avg {frame['avg']} / p95 {frame['p95']} мс; "
          f"прогон {s['duration_s']} с")

## 5. Скачать результаты

JSONL содержит заголовок с метаданными (первая строка) и сводку (последняя).

In [ ]:
from IPython.display import FileLink, display

for out, _summary in results.values():
    display(FileLink(str(out)))